In [ ]:
import numpy as np

from resources.Classes.Simulation_Class import Simulation_Class
from resources.Classes.Wave_vector_class import Wave_vector_class
from resources.Classes.Nbody_classes.Baryonic_N_body import Baryons
from resources.Classes.Nbody_classes.NBodyGas import NBodyGas
from resources.Classes.Nbody_classes.Sink_N_Body import SinkNBody

# ------------------------------------------------------------
# simulation
# ------------------------------------------------------------
m_scalar_eV = 1e-22
MBH = 1e6

boundaries = [[-50, 50], [-50, 50], [-50, 50]]

sim = Simulation_Class(
    dim=3,
    boundaries=boundaries,
    N=128,
    total_time=10.0,
    h=1e-3,
    order_of_evolution=2,
    use_gravity=True,
    static_potential=None,
    save_max_vals=True,
    self_int=False,
    use_sponge=True,
    m_s=m_scalar_eV,
)

# ------------------------------------------------------------
# sink
# ------------------------------------------------------------
sink = SinkNBody(
    simulation=sim,
    N_sinks=1,
    initial_masses=[MBH],
    initial_positions=[[0.0, 0.0, 0.0]],
    initial_velocities=[[0.0, 0.0, 0.0]],
    capture_radius=4.0 * min(sim.dx),
    softening_length=2.0 * min(sim.dx),
)
sim.add_baryons(sink)

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def generate_random_position(boundary):
    low, high = boundary
    return [np.random.uniform(low, high), np.random.uniform(low, high), 0.0]

def is_far_enough(new_pos, existing_positions, min_dist):
    for pos in existing_positions:
        if np.linalg.norm(np.array(new_pos) - np.array(pos)) < min_dist:
            return False
    return True

def make_gas_clump(sim, center, total_mass, core_radius=2.5, rho_floor=1e-12):
    x, y, z = sim.grids
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)

    cx, cy, cz = center
    r2 = (x - cx)**2 + (y - cy)**2 + (z - cz)**2

    # smooth spherical clump
    rho_shape = (1.0 + r2 / core_radius**2)**(-2.5)

    M_current = rho_shape.sum() * sim.dV
    rho = rho_shape * (total_mass / M_current)

    rho += rho_floor
    rho = np.maximum(rho, rho_floor)

    vx = np.zeros_like(rho)
    vy = np.zeros_like(rho)
    vz = np.zeros_like(rho)

    return rho, vx, vy, vz

# ------------------------------------------------------------
# clump parameters
# ------------------------------------------------------------
np.random.seed(1234)

n_clumps = 8
boundary = [-35, 35]
min_separation = 12.0

M_uldm_clump = 3.24389184e8

Nstar_per_clump = int(2.5e4)
Mstar_per_clump = 3.0e6
a_star = 2.5
sigma_star = 5.0
r_trunc_star = 8.0

Mgas_per_clump = 3.0e6
rho_floor = 1e-12
cs = 10.0
gamma = 5/3
tcool = 1.0
cfl = 0.4

# ------------------------------------------------------------
# positions
# ------------------------------------------------------------
positions = []
for i in range(n_clumps):
    while True:
        means = generate_random_position(boundary)
        if is_far_enough(means, positions, min_separation):
            positions.append(means)
            break

# ------------------------------------------------------------
# initialize all clumps
# ------------------------------------------------------------
for i, means in enumerate(positions):
    # ULDM
    vlna = Wave_vector_class(
        packet_type="resources/solitons/GroundState(1).dat",
        means=means,
        st_deviations=[0.5, 0.5, 0.5],
        simulation=sim,
        mass=1,
        omega=1,
        momenta=[0.0, 0.0, 0.0],
        spin=0,
        desired_soliton_mass=M_uldm_clump,
    )
    sim.add_wave_vector(wave_vector=vlna)
    vlna.cleanup_wave_vector()
    del vlna

    # stellar boban
    stars = Baryons(
        simulation=sim,
        N_particles=Nstar_per_clump,
        total_mass=Mstar_per_clump,
        init_profile="hernquist",
        center=tuple(means),
        velocity=(0.0, 0.0, 0.0),
        name=f"nbody{i}",
        vel_sigma=sigma_star,
        scale_radius=a_star,
        truncation_radius=r_trunc_star,
    )
    sim.add_baryons(stars)

    # gas boban
    rho, vx, vy, vz = make_gas_clump(
        sim=sim,
        center=means,
        total_mass=Mgas_per_clump,
        core_radius=2.5,
        rho_floor=rho_floor,
    )

    gas = NBodyGas(
        simulation=sim,
        rho=rho,
        vx=vx,
        vy=vy,
        vz=vz,
        name=f"gas{i}",
        cs=cs,
        gamma=gamma,
        tcool=tcool,
        e_floor=1e-12,
        rho_floor=rho_floor,
        cfl=cfl,
    )
    sim.add_baryons(gas)

# ------------------------------------------------------------
# run
# ------------------------------------------------------------
sim.initialize_simulation()
sim.evolve(save_every=1000)

hbartilde 19.60695038367394


/home/smidmar9/.local/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:479: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


passed mass is 324389184.0
passed mass is 324389184.0
calculated mass is 331812960.0
0.9776266
above is lambda
12.070048
324481842.0410156 sol massss
dynamic mass 322507324.21875
existuju někdy tady?
{0: [<resources.Classes.Wave_function_class.Wave_function object at 0x7f1b6429cd50>]}
tohle končím s len 1 <resources.Classes.Wave_vector_class.Wave_vector_class object at 0x7f1b8e27a650>
a nejspíš bych měl pracovat s len 1 {0: [<resources.Classes.Wave_function_class.Wave_function object at 0x7f1b6429cd50>]}
passed mass is 324389184.0
passed mass is 324389184.0
calculated mass is 331812960.0
0.9776266
above is lambda
12.070048
324334197.9980469 sol massss
dynamic mass 322418548.5839844
nebo dokonce sem
 i je 0
component <resources.Classes.Wave_function_class.Wave_function object at 0x7f1b63fbed10>
{0: [<resources.Classes.Wave_function_class.Wave_function object at 0x7f1b6429cd50>]}
tohle končím s len 1 <resources.Classes.Wave_vector_class.Wave_vector_class object at 0x7f1b6429f0d0>
a nejsp

Simulation Progress:   0%|          | 0/10000 [00:00<?, ?step/s]

starting evolution
is first
[DEBUG W] t=1.000000e-03  W_wave_self=-1.628422e+12  W_wave_sink=-1.726395e+07  W_total=-1.628439e+12
0 0.0 269833600.34178346 134962.63041783904 269968562.9722013
0 0.0 269833892.88934875 134962.66539569356 269968855.5547444
0 0.0 269833669.57030404 134962.6389457995 269968632.20924985
0 0.0 269833384.98050517 134962.60097483985 269968347.58148
0 0.0 269833580.29099095 134962.62813868112 269968542.9191296
0 0.0 269833644.1504001 134962.63555615695 269968606.78595626
0 0.0 269833706.49262124 134962.6384092058 269968669.13103044
0 0.0 269833688.5271354 134962.64043227563 269968651.1675677


Simulation Progress:   0%|          | 1/10000 [00:01<4:48:31,  1.73s/step]

... (46 outputs left out)

8000 8.0 816879922.8003308 14211494371.308714 15028374294.109045
8000 8.0 1093178371.8350067 14398527594.790123 15491705966.62513
8000 8.0 1375905297.091908 12234389154.58523 13610294451.677137
8000 8.0 905573279.8859057 16984843286.148102 17890416566.034008
8000 8.0 1014038990.9702028 15970565155.82461 16984604146.794813
8000 8.0 908067504.051322 17882001838.244457 18790069342.29578
8000 8.0 811717566.0297292 10177665671.71528 10989383237.74501
8000 8.0 1540637996.7578807 17499551507.393703 19040189504.151585


Simulation Progress:  80%|████████  | 8001/10000 [56:37<26:13,  1.27step/s]

[Gas invariants] t=8.0010  dM=3.104408582051e-16  dP=(7.118248417481e+03,-1.922477588147e+04,-1.448753761848e+03)


Simulation Progress:  81%|████████  | 8082/10000 [57:10<12:41,  2.52step/s]

# import subprocess, sys, os
from scipy.integrate import odeint
import matplotlib.pyplot as plt

cmd = [sys.executable, "ModosFinder.py"]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

import time
time.sleep(1)
print("poll =", p.poll())   # None = pořád běží, číslo = exit code

import os, sys
print("marimo cwd:", os.getcwd())
print("python:", sys.executable)
print("ModoFinder exists here?:", os.path.exists("ModosFinder.py"))


# vytiskni pár řádků výstupu:
for _ in range(20):
    line = p.stdout.readline()
    if not line:
        break
    print(line, end="")


ModuleNotFoundError: No module named 'resources.Functions.reuse_simulation_output'